# Email Thread Summarizer

## 1. Project Overview

**Task:** Condense long email chains into structured summaries capturing participants, decisions, action items, and the final resolution.

**Approach:** Parse email thread structure, then use decomposed prompts to extract decisions, action items, open questions, and a concise narrative summary. We compare rule-based extraction against LLM-based extraction.

**Stack:**
- `LangChain` + `ChatOllama` — local LLM interaction
- `Ollama` + `qwen3.5:9b` — local LLM (no API keys)

> **No API keys required.** Everything runs locally.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | **Parse** email thread structure (sender, date, body, reply chain) |
| 2 | **Extract** decisions, action items, and open questions from threads |
| 3 | **Summarize** multi-turn conversations into concise digests |
| 4 | Handle **forwarded** and **CC'd** emails |
| 5 | Compare **rule-based** vs **LLM-based** extraction |

## 3. Problem Statement

Email threads with 5–20+ messages are common in business. Key information gets buried. Readers need:
- Who said what (participant tracking)
- What was decided (decisions)
- What needs to happen next (action items)
- What's still unresolved (open questions)

## 4. Why This Matters

- **Workplace productivity:** Average professional spends 28% of work time on email
- **Onboarding:** New team members can catch up on decisions without reading entire threads
- **Audit trail:** Structured summaries make decisions searchable and traceable

## 5. Environment Setup

In [ ]:
import os, json, re, time, warnings
from textwrap import dedent
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

LLM_MODEL = "qwen3.5:9b"
llm = ChatOllama(model=LLM_MODEL, temperature=0)

def clean(text):
    if "<think>" in text: text = text.split("</think>")[-1].strip()
    return text.strip()

def ask(system, user):
    return clean(llm.invoke([SystemMessage(content=system), HumanMessage(content=user)]).content)

def parse_json(text):
    text = clean(text)
    if "```" in text: text = re.sub(r"```(?:json)?\s*", "", text).replace("```", "")
    start, end = text.find("{"), text.rfind("}") + 1
    if start >= 0 and end > start:
        try: return json.loads(text[start:end])
        except json.JSONDecodeError: pass
    start, end = text.find("["), text.rfind("]") + 1
    if start >= 0 and end > start:
        try: return json.loads(text[start:end])
        except json.JSONDecodeError: pass
    return None

print(f"LLM ready: {ask('Reply with one word.', 'Say ready.')[:20]}")

---

## 6. Sample Email Threads

We create two realistic email threads of different complexity.

In [ ]:
THREADS = {
    "budget_approval": {
        "subject": "Q2 Marketing Budget Approval",
        "emails": [
            {"from": "alice@company.com", "to": "team@company.com", "date": "2026-03-15 09:00",
             "body": "Hi team, we need to finalize the Q2 marketing budget by Friday. Please review the attached spreadsheet and share your department estimates. Total available: $200K."},
            {"from": "bob@company.com", "to": "team@company.com", "date": "2026-03-15 10:30",
             "body": "Thanks Alice. Digital advertising: $80K (up from $60K last quarter due to new product launch). Content creation: $25K. I'll send the detailed breakdown by EOD."},
            {"from": "carol@company.com", "to": "team@company.com", "date": "2026-03-15 11:15",
             "body": "Events budget: $45K for the annual conference + two regional meetups. This is firm — venue deposits are already paid ($12K non-refundable). PR agency retainer: $20K/quarter."},
            {"from": "alice@company.com", "to": "team@company.com", "date": "2026-03-15 14:00",
             "body": "Running total: Digital $80K + Content $25K + Events $45K + PR $20K = $170K. That leaves $30K buffer. Carol, can the PR retainer be reduced to $15K? Bob, is $80K justified for digital?"},
            {"from": "carol@company.com", "to": "team@company.com", "date": "2026-03-16 09:15",
             "body": "I spoke with the agency. They can do $17K if we commit for two quarters. That's the best they'll offer."},
            {"from": "bob@company.com", "to": "team@company.com", "date": "2026-03-16 10:00",
             "body": "I can reduce digital to $75K by cutting the TikTok experiment ($5K). The core Google and Meta campaigns are non-negotiable for the product launch."},
            {"from": "alice@company.com", "to": "team@company.com", "date": "2026-03-16 11:30",
             "body": "Final budget: Digital $75K + Content $25K + Events $45K + PR $17K = $162K. Buffer: $38K. I'll submit to finance today. Bob, please send the detailed digital breakdown by EOD. Carol, get the agency contract signed for two quarters at $17K."},
        ],
        "ground_truth": {
            "decisions": ["Total Q2 budget set at $162K of $200K available", "Cut TikTok experiment to reduce digital to $75K", "PR retainer negotiated to $17K for two-quarter commitment"],
            "action_items": [{"owner": "Bob", "task": "Send detailed digital breakdown", "deadline": "EOD"}, {"owner": "Carol", "task": "Sign agency contract for two quarters at $17K"}, {"owner": "Alice", "task": "Submit budget to finance", "deadline": "today"}],
        },
    },
    "vendor_selection": {
        "subject": "CRM Vendor Selection - Final Decision Needed",
        "emails": [
            {"from": "david@company.com", "to": "steering@company.com", "date": "2026-03-20 08:00",
             "body": "Team, we need to make a final CRM decision this week. We've narrowed it to Salesforce ($180K/yr), HubSpot ($95K/yr), and Pipedrive ($42K/yr). Each has completed demos. Please share your recommendation with rationale."},
            {"from": "elena@company.com", "to": "steering@company.com", "date": "2026-03-20 10:00",
             "body": "From Sales perspective: Salesforce is the most feature-rich but complex. Training would take 3 months. HubSpot covers 90% of our needs with much less training (2 weeks). I recommend HubSpot."},
            {"from": "frank@company.com", "to": "steering@company.com", "date": "2026-03-20 11:30",
             "body": "IT perspective: HubSpot integrates better with our existing stack (Google Workspace, Slack, Stripe). Salesforce would need a dedicated admin ($90K/yr salary). Pipedrive lacks API depth for our custom reporting needs. I also recommend HubSpot."},
            {"from": "grace@company.com", "to": "steering@company.com", "date": "2026-03-20 14:00",
             "body": "Finance perspective: Total cost of ownership over 3 years — Salesforce: $810K (license + admin + training), HubSpot: $315K, Pipedrive: $126K. HubSpot is the best value for our scale. However, if we grow past 200 users, Salesforce becomes more cost-effective per seat."},
            {"from": "david@company.com", "to": "steering@company.com", "date": "2026-03-21 09:00",
             "body": "Consensus is clear: HubSpot. I'll negotiate the contract this week. Elena, please draft the migration plan. Frank, set up the sandbox environment. Grace, process the PO. Target go-live: May 1st."},
        ],
        "ground_truth": {
            "decisions": ["Selected HubSpot as CRM vendor", "Target go-live May 1st"],
            "action_items": [{"owner": "David", "task": "Negotiate HubSpot contract"}, {"owner": "Elena", "task": "Draft migration plan"}, {"owner": "Frank", "task": "Set up sandbox environment"}, {"owner": "Grace", "task": "Process purchase order"}],
        },
    },
}

for key, thread in THREADS.items():
    print(f"{key}: {thread['subject']} | {len(thread['emails'])} emails")

## 7. Rule-Based Thread Parsing

In [ ]:
def parse_thread(thread):
    participants = list(set(e["from"].split("@")[0] for e in thread["emails"]))
    date_range = f"{thread['emails'][0]['date']} to {thread['emails'][-1]['date']}"
    return {"subject": thread["subject"], "participants": participants, "email_count": len(thread["emails"]), "date_range": date_range}

for key, thread in THREADS.items():
    info = parse_thread(thread)
    print(f"\n{info['subject']}")
    print(f"  Participants: {', '.join(info['participants'])}")
    print(f"  Emails: {info['email_count']} | Dates: {info['date_range']}")

## 8. LLM-Based Thread Summarization

In [ ]:
THREAD_SUMMARY_SYSTEM = """You summarize email threads for busy professionals.
Extract:
1. A 3-4 sentence narrative summary
2. Key decisions made
3. Action items with owners
4. Open questions (if any)

Return JSON:
{"summary": "narrative", "decisions": ["d1", "d2"], "action_items": [{"owner": "Name", "task": "desc"}], "open_questions": ["q1"]}"""

def format_thread_for_llm(thread):
    lines = [f"Subject: {thread['subject']}\n"]
    for e in thread["emails"]:
        lines.append(f"From: {e['from']} | Date: {e['date']}")
        lines.append(f"{e['body']}\n")
    return "\n".join(lines)

all_results = {}
for key, thread in THREADS.items():
    formatted = format_thread_for_llm(thread)
    t0 = time.perf_counter()
    resp = ask(THREAD_SUMMARY_SYSTEM, f"Email thread:\n{formatted}\n\nExtract:")
    elapsed = time.perf_counter() - t0
    parsed = parse_json(resp)
    if parsed is None: parsed = {"raw": resp[:500]}
    parsed["_time"] = round(elapsed, 1)
    all_results[key] = parsed

    print(f"\n{'=' * 60}")
    print(f"Thread: {thread['subject']} ({elapsed:.1f}s)")
    print(f"{'=' * 60}")
    print(f"\nSummary: {parsed.get('summary', 'N/A')}")
    print(f"\nDecisions:")
    for d in parsed.get("decisions", []): print(f"  \u2705 {d}")
    print(f"\nAction Items:")
    for a in parsed.get("action_items", []):
        if isinstance(a, dict): print(f"  \u2022 [{a.get('owner','?')}] {a.get('task','?')}")
    if parsed.get("open_questions"):
        print(f"\nOpen Questions:")
        for q in parsed["open_questions"]: print(f"  \u2753 {q}")

## 9. Evaluation Against Ground Truth

In [ ]:
for key, thread in THREADS.items():
    gt = thread["ground_truth"]
    pred = all_results[key]
    print(f"\n{thread['subject']}")
    print(f"  Decisions  — GT: {len(gt['decisions'])}, Extracted: {len(pred.get('decisions', []))}")
    print(f"  Actions    — GT: {len(gt['action_items'])}, Extracted: {len(pred.get('action_items', []))}")

## 10. Output Formatting

In [ ]:
def format_email_summary(thread, result):
    info = parse_thread(thread)
    lines = [f"# Thread Summary: {info['subject']}",
             f"**Participants:** {', '.join(info['participants'])}",
             f"**Emails:** {info['email_count']} | **Period:** {info['date_range']}\n",
             "## Summary", result.get("summary", "N/A"), "",
             "## Decisions"]
    for d in result.get("decisions", []): lines.append(f"- \u2705 {d}")
    lines.append("\n## Action Items")
    lines.append("| Owner | Task |")
    lines.append("|-------|------|")
    for a in result.get("action_items", []):
        if isinstance(a, dict): lines.append(f"| {a.get('owner','?')} | {a.get('task','?')} |")
    return "\n".join(lines)

for key, thread in THREADS.items():
    print(format_email_summary(thread, all_results[key]))
    print("\n" + "=" * 60)

## 11. Edge Cases

In [ ]:
# Edge case: Thread with disagreement and no resolution
unresolved = {"subject": "Office Reopening Policy", "emails": [
    {"from": "hr@co.com", "to": "all@co.com", "date": "2026-04-01 09:00", "body": "We're proposing 3 days/week in-office starting May. Please share feedback."},
    {"from": "eng-lead@co.com", "to": "all@co.com", "date": "2026-04-01 10:00", "body": "Engineering strongly opposes. Our productivity metrics are 15% higher remote. We propose 1 day/week max."},
    {"from": "sales-lead@co.com", "to": "all@co.com", "date": "2026-04-01 11:00", "body": "Sales supports 3 days. Client meetings require in-person presence. We've lost two deals due to remote-only."},
    {"from": "hr@co.com", "to": "all@co.com", "date": "2026-04-01 15:00", "body": "No consensus yet. Let's schedule a town hall for Friday to discuss further."},
]}

resp = ask(THREAD_SUMMARY_SYSTEM, f"Email thread:\n{format_thread_for_llm(unresolved)}\n\nExtract:")
parsed = parse_json(resp)
print("EDGE CASE: Unresolved thread")
print(f"  Summary: {parsed.get('summary', resp[:200]) if parsed else resp[:200]}")
print(f"  Open questions: {parsed.get('open_questions', []) if parsed else 'N/A'}")
print("  \u2192 Should identify the lack of consensus and the scheduled follow-up")

## 12. Common Mistakes & Key Takeaways

| Mistake | Fix |
|---------|-----|
| Treating all emails equally | Later emails often carry decisions; weight them |
| Missing implicit decisions | "Let's go with X" is a decision even without "decided" |
| Not tracking open questions | Unresolved threads need open-question extraction |

| # | Takeaway |
|---|----------|
| 1 | **Thread structure matters** — parse sender, date, and reply chain before summarizing |
| 2 | **Decomposed extraction** beats a single "summarize this" prompt |
| 3 | **Open questions** are as important as decisions for thread summaries |
| 4 | **Later emails carry more weight** — they contain decisions and resolutions |
| 5 | **Format for the consumer** — Slack, email digest, or project tracker |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*